In [1]:
!pip install redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.9/268.9 kB 6.3 MB/s eta 0:00:0000:01


In [2]:
import pandas as pd
import redis

In [12]:
dr = redis.Redis(host='redis', port=6379, db=0)

In [15]:
print("Part 1 – Data Exploration")

Part 1 – Data Exploration


In [14]:
# 1. How many actors and movies are stored in Redis?
num_actors = len(list(dr.scan_iter("actor:*")))
num_actors

1319

In [17]:
num_movies = len(list(dr.scan_iter("movie:*")))
num_movies

922

In [18]:
print(f"Number of actors: {num_actors}")
print(f"Number of movies: {num_movies}")

Number of actors: 1319
Number of movies: 922


In [19]:
# 2. List 5 actors born before 1980
actors_born_before_1980 = []
for key in dr.scan_iter("actor:*"):
    dob = dr.hget(key, "date_of_birth")
    try:
        if int(dob) < 1980:
            actor_data = dr.hgetall(key)
            actors_born_before_1980.append(actor_data)
    except:
        continue
    if len(actors_born_before_1980) >= 5:
        break
print("Actors born before 1980:")
for actor in actors_born_before_1980:
    print(actor)

Actors born before 1980:
{b'first_name': b'Naomi', b'last_name': b'Ryan', b'date_of_birth': b'1977'}
{b'first_name': b'Alicia', b'last_name': b'Silverstone', b'date_of_birth': b'1976'}
{b'first_name': b'Cory', b'last_name': b'Hardrict', b'date_of_birth': b'1979'}
{b'first_name': b'Ward', b'last_name': b'Horton', b'date_of_birth': b'1976'}
{b'first_name': b'Naomi', b'last_name': b'Watts', b'date_of_birth': b'1968'}


In [41]:
# 3. Retrieve the genre and rating of the movie "The Imitation Game"
for key in dr.scan_iter("movie:*"):
    data = dr.hgetall(key)
    if data.get("title".encode('utf-8')).decode('utf-8') == "The Imitation Game":
        print(f"Genre: {data.get('genre'.encode('utf-8'))}, Rating: {data.get('rating'.encode('utf-8'))}")
        break

Genre: b'Biography', Rating: b'8.1'


8

In [42]:
# 4. List the top 5 highest-rated movies
movie_list = []
for key in dr.scan_iter("movie:*"):
    data = dr.hgetall(key)
    try:
        rating = float(data.get("rating".encode('utf-8'), 0))
        title = data.get("title".encode('utf-8'), "Unknown")
        movie_list.append((title, rating))
    except:
        continue
top_5_movies = sorted(movie_list, key=lambda x: x[1], reverse=True)[:5]
print("Top 5 highest-rated movies:")
for title, rating in top_5_movies:
    print(f"{title} - {rating}")

Top 5 highest-rated movies:
b'Boy 9' - 9.4
b'Vegas (doc)' - 9.4
b'The Shawshank Redemption' - 9.3
b'Ween Live in Chicago' - 9.2
b'Over Canada: An Aerial Adventure' - 9.1


In [45]:
# 5. How many movies have a rating above 7.5?
count_high_rating = sum(
    1 for key in dr.scan_iter("movie:*")
    if float(dr.hget(key, "rating") or 0) > 7.5
)
print(f"Movies with rating > 7.5: {count_high_rating}")

Movies with rating > 7.5: 183


In [46]:
# 6. Update the rating of "The Imitation Game" to 8.5
for key in dr.scan_iter("movie:*"):
    if dr.hget(key, "title").decode('utf-8') == "The Imitation Game":
        dr.hset(key, "rating", 8.5)
        print("Rating updated to 8.5")
        break

Rating updated to 8.5


In [47]:
# 7. Add new actor: Zendaya, born in 1996
new_actor_id = num_actors + 1
dr.hset(f"actor:{new_actor_id}", 
        mapping={
            "first_name": "Zendaya", 
            "last_name": "", 
            "date_of_birth": "1996"
        }
)
print(f"Added new actor Zendaya with ID actor:{new_actor_id}")


Added new actor Zendaya with ID actor:1320


In [49]:
# 8. Delete movie titled "The Room"
for key in dr.scan_iter("movie:*"):
    if dr.hget(key, "title").decode('utf-8') == "The Room":
        dr.delete(key)
        print(f"Deleted movie: The Room ({key})")
        break


In [50]:
print("\nPart 2 – Advanced Queries")


Part 2 – Advanced Queries


In [51]:
# 1. Count actors with last name starting with "P"
count_p = 0
for key in dr.scan_iter("actor:*"):
    lname = dr.hget(key, "last_name").decode('utf-8')
    if lname and lname.startswith("P"):
        count_p += 1
print(f"Actors with last name starting with P: {count_p}")

Actors with last name starting with P: 64


In [55]:
# 2. Movies released after 2010 with more than 100,000 votes
selected_movies = []
for key in dr.scan_iter("movie:*"):
    data = dr.hgetall(key)
    try:
        year = int(data.get("release_year".encode('utf-8'), 0))
        votes = int(data.get("votes".encode('utf-8'), 0))
        if year > 2010 and votes > 100000:
            selected_movies.append(data.get("title".encode('utf-8'), 0))
    except:
        continue
print("Movies after 2010 with > 100000 votes (sample of 10):")
for m in selected_movies[:10]:
    print(m)

Movies after 2010 with > 100000 votes (sample of 10):
b'Deadpool'
b'Insurgent'
b'Straight Outta Compton'
b'John Wick'
b'Guardians of the Galaxy'
b'The Revenant'
b'Jack Reacher: Never Go Back'
b'Maze Runner: The Scorch Trials'
b'The Imitation Game'
b'The Amazing Spider-Man 2'


In [57]:
# 3. Create hash: top_movies_by_genre:<genre> with top-rated movie per genre
top_by_genre = {}
for key in dr.scan_iter("movie:*"):
    data = dr.hgetall(key)
    genre = data.get("genre".encode('utf-8'))
    try:
        rating = float(data.get("rating".encode('utf-8'), 0))
    except:
        continue
    if not genre:
        continue
    if genre not in top_by_genre or rating > top_by_genre[genre]["rating"]:
        top_by_genre[genre] = {"title": data.get("title".encode('utf-8')), "rating": rating}

# Save in Redis
for genre, info in top_by_genre.items():
    dr.hset(f"top_movies_by_genre:{genre}", mapping={"title": info["title"], "rating": info["rating"]})
    print(f"Stored top movie for genre {genre}: {info['title']} with rating {info['rating']}")

Stored top movie for genre b'Comedy': b'Pretty Village,Pretty Flame' with rating 8.7
Stored top movie for genre b'Action': b'The Matrix' with rating 8.7
Stored top movie for genre b'Crime': b'Pulp Finction' with rating 8.9
Stored top movie for genre b'Animation': b'Future Boy Conan' with rating 8.7
Stored top movie for genre b'Adventure': b'Interstellar' with rating 8.6
Stored top movie for genre b'Biography': b'Hacksaw Ridge' with rating 8.8
Stored top movie for genre b'Horror': b'Split' with rating 8.1
Stored top movie for genre b'Drama': b'The Shawshank Redemption' with rating 9.3
Stored top movie for genre b'Thriller': b'The Sleep of Reason' with rating 6.7
Stored top movie for genre b'Music': b'Heart: Alive in Seattle' with rating 8.6
Stored top movie for genre b'Short': b'Boy 9' with rating 9.4
Stored top movie for genre b'Documentary': b'Vegas (doc)' with rating 9.4
Stored top movie for genre b'Reality-TV': b'Come Dine with Me Canada' with rating 7.4
Stored top movie for genre b